#Инициализация зависимостей

In [ ]:
# cell 1: Установка и импорт

!pip install sentence-transformers faiss-cpu langchain-text-splitters scikit-learn pandas numpy matplotlib seaborn joblib

import os
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter, defaultdict
import pickle
import joblib
import warnings
from google.colab import drive

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings('ignore')

drive.mount('/content/drive')

print("All libraries loaded")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 30.8 MB/s eta 0:00:00
Mounted at /content/drive
All libraries loaded
PyTorch: 2.10.0+cpu
CUDA available: False


#cell 2


In [ ]:
# cell 2: Классы RAGFeatureExtractor и OrderPredictionRAG

from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
import time
import os
import pickle
import joblib
import faiss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer


class RAGFeatureExtractor:
    def __init__(self, embedding_model_name='intfloat/multilingual-e5-small'):
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.embedding_dim = self.embedding_model.get_sentence_embedding_dimension()

        self.feature_stores = {
            'previous_orders': {'texts': [], 'embeddings': None, 'index': None},
            'current_context': {'texts': [], 'embeddings': None, 'index': None},
            'user_data': {'texts': [], 'embeddings': None, 'index': None},
            'calendar': {'texts': [], 'embeddings': None, 'index': None},
            'cart': {'texts': [], 'embeddings': None, 'index': None}
        }

        print(f"RAG Feature Extractor initialized")
        print(f"  Embedding dimension: {self.embedding_dim}")

    def text_to_embedding(self, texts: List[str]) -> np.ndarray:
        return self.embedding_model.encode(
            texts,
            show_progress_bar=False,
            batch_size=32,
            normalize_embeddings=True
        )

    def build_index(self, source_name: str, texts: List[str]):
        if source_name not in self.feature_stores:
            raise ValueError(f"Unknown source: {source_name}")

        print(f"  Building index for {source_name}...")
        embeddings = self.text_to_embedding(texts)

        self.feature_stores[source_name]['texts'] = texts
        self.feature_stores[source_name]['embeddings'] = embeddings

        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings.astype('float32'))
        self.feature_stores[source_name]['index'] = index

        print(f"    {source_name}: {len(texts)} records")

    def build_all_indices(self, data: Dict[str, List[str]]):
        print("\nBuilding indices for all sources...")
        for source_name, texts in data.items():
            if source_name in self.feature_stores and texts:
                self.build_index(source_name, texts)

    def extract_features_for_query(self, query_features: Dict[str, str], k: int = 5) -> np.ndarray:
        feature_vector = []

        for source_name in self.feature_stores.keys():
            query_text = query_features.get(source_name, '')
            if not query_text or self.feature_stores[source_name]['index'] is None:
                feature_vector.extend([0] * self.embedding_dim)
                continue

            query_emb = self.text_to_embedding([query_text])

            similarities, indices = self.feature_stores[source_name]['index'].search(
                query_emb.astype('float32'), k
            )

            feature_vector.extend(query_emb[0].tolist())

            valid_sims = similarities[0][similarities[0] > -1]
            max_sim = valid_sims.max() if len(valid_sims) > 0 else 0
            mean_sim = valid_sims.mean() if len(valid_sims) > 0 else 0
            feature_vector.extend([max_sim, mean_sim])

            retrieved_texts = []
            for idx in indices[0]:
                if idx != -1 and idx < len(self.feature_stores[source_name]['texts']):
                    retrieved_texts.append(self.feature_stores[source_name]['texts'][idx])

            if retrieved_texts:
                retrieved_embs = self.text_to_embedding(retrieved_texts)
                mean_retrieved = retrieved_embs.mean(axis=0)
                feature_vector.extend(mean_retrieved.tolist())
            else:
                feature_vector.extend([0] * self.embedding_dim)

        return np.array(feature_vector)

    def extract_batch_features(self, queries: List[Dict[str, str]], k: int = 5) -> np.ndarray:
        features = []
        for query in queries:
            features.append(self.extract_features_for_query(query, k))
        return np.array(features)


class OrderPredictionRAG:
    def __init__(self, embedding_model='intfloat/multilingual-e5-small'):
        self.feature_extractor = RAGFeatureExtractor(embedding_model)
        self.rf_classifier = None
        self.label_encoder = LabelEncoder()
        self.product_catalog = []
        self.scaler = StandardScaler()

    def prepare_training_data(self, training_data: pd.DataFrame):
        print("\nPreparing training data...")

        sources_data = {
            'previous_orders': training_data['previous_orders_text'].tolist(),
            'current_context': training_data['current_context_text'].tolist(),
            'user_data': training_data['user_data_text'].tolist(),
            'calendar': training_data['calendar_text'].tolist(),
            'cart': training_data['cart_text'].tolist()
        }

        self.feature_extractor.build_all_indices(sources_data)

        all_products = set()
        for targets in training_data['target_positions']:
            if isinstance(targets, list):
                all_products.update(targets)
            else:
                all_products.add(targets)
        self.product_catalog = sorted(list(all_products))

        print(f"  Product catalog: {len(self.product_catalog)} unique items")
        return self.product_catalog

    def train(self, training_data: pd.DataFrame, n_estimators=100, max_depth=15):
        print("\nTRAINING RAG + RANDOM FOREST")
        print("=" * 50)

        self.prepare_training_data(training_data)

        print("\nExtracting RAG features...")
        queries = []
        for _, row in training_data.iterrows():
            query = {
                'previous_orders': row['previous_orders_text'],
                'current_context': row['current_context_text'],
                'user_data': row['user_data_text'],
                'calendar': row['calendar_text'],
                'cart': row['cart_text']
            }
            queries.append(query)

        X = self.feature_extractor.extract_batch_features(queries)
        print(f"  Feature space size: {X.shape}")

        X = self.scaler.fit_transform(X)

        y = []
        for _, row in training_data.iterrows():
            targets = row['target_positions']
            if isinstance(targets, list):
                y.append('|'.join(targets))
            else:
                y.append(targets)

        y_encoded = self.label_encoder.fit_transform(y)
        print(f"  Number of classes: {len(self.label_encoder.classes_)}")

        print("\nTraining Random Forest...")
        self.rf_classifier = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )

        start_time = time.time()
        self.rf_classifier.fit(X, y_encoded)
        train_time = time.time() - start_time

        print(f"  Training completed in {train_time:.2f} seconds")

        importance = self.rf_classifier.feature_importances_
        print("\nTop 10 feature importances:")
        top_idx = np.argsort(importance)[-10:][::-1]
        for i, idx in enumerate(top_idx[:10]):
            print(f"  {i+1}. Feature {idx}: {importance[idx]:.4f}")

        return self.rf_classifier

    def predict(self, query_features: Dict[str, str], top_k: int = 5) -> List[str]:
        if self.rf_classifier is None:
            raise ValueError("Model not trained. Call train() first.")

        X = self.feature_extractor.extract_features_for_query(query_features)
        X = X.reshape(1, -1)
        X = self.scaler.transform(X)

        pred_encoded = self.rf_classifier.predict(X)[0]
        pred_label = self.label_encoder.inverse_transform([pred_encoded])[0]

        if '|' in pred_label:
            return pred_label.split('|')[:top_k]
        return [pred_label]

    def predict_with_proba(self, query_features: Dict[str, str], top_k: int = 5) -> List[Tuple[str, float]]:
        X = self.feature_extractor.extract_features_for_query(query_features)
        X = X.reshape(1, -1)
        X = self.scaler.transform(X)

        probas = self.rf_classifier.predict_proba(X)[0]
        top_indices = np.argsort(probas)[-top_k:][::-1]

        results = []
        for idx in top_indices:
            label = self.label_encoder.inverse_transform([idx])[0]
            if '|' in label:
                products = label.split('|')
                for product in products[:top_k]:
                    results.append((product, probas[idx] / len(products)))
            else:
                results.append((label, probas[idx]))

        aggregated = {}
        for product, prob in results:
            aggregated[product] = max(aggregated.get(product, 0), prob)

        return sorted(aggregated.items(), key=lambda x: x[1], reverse=True)[:top_k]

    def evaluate(self, test_data: pd.DataFrame):
        print("\nMODEL EVALUATION")
        print("=" * 50)

        predictions = []
        true_labels = []

        for _, row in test_data.iterrows():
            query = {
                'previous_orders': row['previous_orders_text'],
                'current_context': row['current_context_text'],
                'user_data': row['user_data_text'],
                'calendar': row['calendar_text'],
                'cart': row['cart_text']
            }

            pred = self.predict(query)
            predictions.append(pred[0] if pred else 'unknown')

            target = row['target_positions']
            if isinstance(target, list):
                true_labels.append(target[0] if target else 'unknown')
            else:
                true_labels.append(target)

        accuracy = accuracy_score(true_labels, predictions)
        print(f"\nAccuracy: {accuracy:.4f}")

        print("\nClassification Report:")
        print(classification_report(true_labels, predictions, zero_division=0))

        return {
            'accuracy': accuracy,
            'predictions': predictions,
            'true_labels': true_labels
        }

    def save_model(self, path='/content/drive/MyDrive/rag_order_model'):
        os.makedirs(path, exist_ok=True)

        model_data = {
            'rf_classifier': self.rf_classifier,
            'label_encoder': self.label_encoder,
            'product_catalog': self.product_catalog,
            'scaler': self.scaler
        }
        joblib.dump(model_data, f"{path}/model.pkl")

        for source_name, store in self.feature_extractor.feature_stores.items():
            if store['index'] is not None:
                faiss.write_index(store['index'], f"{path}/faiss_{source_name}.bin")
                with open(f"{path}/texts_{source_name}.pkl", 'wb') as f:
                    pickle.dump(store['texts'], f)

        print(f"Model saved to {path}")

    def load_model(self, path='/content/drive/MyDrive/rag_order_model'):
        model_data = joblib.load(f"{path}/model.pkl")
        self.rf_classifier = model_data['rf_classifier']
        self.label_encoder = model_data['label_encoder']
        self.product_catalog = model_data['product_catalog']
        self.scaler = model_data['scaler']

        for source_name in self.feature_extractor.feature_stores:
            index_path = f"{path}/faiss_{source_name}.bin"
            texts_path = f"{path}/texts_{source_name}.pkl"

            if os.path.exists(index_path) and os.path.exists(texts_path):
                self.feature_extractor.feature_stores[source_name]['index'] = faiss.read_index(index_path)
                with open(texts_path, 'rb') as f:
                    self.feature_extractor.feature_stores[source_name]['texts'] = pickle.load(f)

        print(f"Model loaded from {path}")

#Класс для работы с признаками


In [ ]:
# cell 3: Загрузка ZIP архива с данными (исправленная)

import pandas as pd
import os
import zipfile
from google.colab import drive, files

drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/instacart_data/'
os.makedirs(DATA_PATH, exist_ok=True)

print("=" * 60)
print("INSTACART DATASET LOADER")
print("=" * 60)

# --------------------------------------------------------------------------
# Проверка, есть ли уже данные в Drive
# --------------------------------------------------------------------------

csv_files_in_drive = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]

if len(csv_files_in_drive) >= 4:
    print(f"\nFound existing data in {DATA_PATH}")
    print(f"Files: {csv_files_in_drive}")
    use_existing = input("\nUse existing data? (y/n): ").lower() == 'y'
else:
    use_existing = False

# --------------------------------------------------------------------------
# Если данных нет или пользователь хочет загрузить новые
# --------------------------------------------------------------------------

if not use_existing:
    print("\nPlease select the Instacart ZIP file you downloaded:")
    uploaded = files.upload()

    zip_file = None
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            zip_file = filename
            break

    if zip_file is None:
        raise Exception("No ZIP file uploaded")

    print(f"\nUploaded: {zip_file}")

    print("\nExtracting files...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(DATA_PATH)

    print(f"Extracted to: {DATA_PATH}")

    os.remove(zip_file)
    print(f"Removed zip file: {zip_file}")

# --------------------------------------------------------------------------
# Сканирование и загрузка CSV файлов (с правильным сопоставлением)
# --------------------------------------------------------------------------

print("\nScanning for CSV files:")
all_csv_files = []
for f in os.listdir(DATA_PATH):
    if f.endswith('.csv'):
        file_path = os.path.join(DATA_PATH, f)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        all_csv_files.append(f)
        print(f"  {f}: {size_mb:.1f} MB")

# Точное сопоставление имен файлов
def load_exact_file(filename_pattern):
    for f in all_csv_files:
        if filename_pattern.lower() == f.lower():
            full_path = os.path.join(DATA_PATH, f)
            df = pd.read_csv(full_path)
            print(f"  Loaded {filename_pattern}: {len(df):,} rows, {len(df.columns)} columns")
            return df
    return None

def load_contains(pattern):
    for f in all_csv_files:
        if pattern.lower() in f.lower():
            full_path = os.path.join(DATA_PATH, f)
            df = pd.read_csv(full_path)
            print(f"  Loaded {pattern}: {len(df):,} rows, {len(df.columns)} columns")
            return df
    print(f"  {pattern}: not found")
    return None

# Загружаем правильные файлы
aisles = load_exact_file('aisles.csv')
departments = load_exact_file('departments.csv')
products = load_exact_file('products.csv')
orders = load_exact_file('orders.csv')

# order_products может иметь разные имена
order_products = load_exact_file('order_products__prior.csv')
if order_products is None:
    order_products = load_exact_file('order_products_train.csv')
if order_products is None:
    order_products = load_contains('order_product')

# --------------------------------------------------------------------------
# Проверка, что загружены правильные файлы
# --------------------------------------------------------------------------

print("\n" + "=" * 60)
print("VERIFYING DATA STRUCTURE")
print("=" * 60)

if products is not None:
    print(f"products table columns: {products.columns.tolist()}")
    if 'department_id' in products.columns:
        print("  CORRECT: products table has department_id")
    else:
        print("  ERROR: products table missing department_id")
        print("  This should be the product catalog, not order_products")

if orders is not None:
    print(f"orders table columns: {orders.columns.tolist()}")
    if 'user_id' in orders.columns:
        print("  CORRECT: orders table has user_id")

if order_products is not None:
    print(f"order_products table columns: {order_products.columns.tolist()}")
    if 'order_id' in order_products.columns and 'product_id' in order_products.columns:
        print("  CORRECT: order_products table has order_id and product_id")

# --------------------------------------------------------------------------
# Итоговая информация
# --------------------------------------------------------------------------

print("\n" + "=" * 60)
print("DATA SUMMARY")
print("=" * 60)

if aisles is not None:
    print(f"aisles:           {len(aisles):,}")
if departments is not None:
    print(f"departments:      {len(departments):,}")
if products is not None:
    print(f"products:         {len(products):,}")
if orders is not None:
    print(f"orders:           {len(orders):,}")
if order_products is not None:
    print(f"order_products:   {len(order_products):,}")

print("=" * 60)

# Проверка целостности
if products is not None and 'department_id' not in products.columns:
    print("\nWARNING: products table is missing 'department_id' column")
    print("This means products is actually order_products or wrong file")
    print("\nCheck the files in:", DATA_PATH)
    print("Expected files: aisles.csv, departments.csv, products.csv, orders.csv, order_products__prior.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INSTACART DATASET LOADER

Found existing data in /content/drive/MyDrive/instacart_data/
Files: ['aisles.csv', 'departments.csv', 'order_products__prior.csv', 'order_products__train.csv', 'orders.csv', 'products.csv']

Use existing data? (y/n): y

Scanning for CSV files:
  aisles.csv: 0.0 MB
  departments.csv: 0.0 MB
  order_products__prior.csv: 550.8 MB
  order_products__train.csv: 23.5 MB
  orders.csv: 103.9 MB
  products.csv: 2.1 MB
  Loaded aisles.csv: 134 rows, 2 columns
  Loaded departments.csv: 21 rows, 2 columns
  Loaded products.csv: 49,688 rows, 4 columns
  Loaded orders.csv: 3,421,083 rows, 7 columns
  Loaded order_products__prior.csv: 32,434,489 rows, 4 columns

VERIFYING DATA STRUCTURE
products table columns: ['product_id', 'product_name', 'aisle_id', 'department_id']
  CORRECT: products table has department_id
orders table columns: ['order_id', '

##Сгрузка датасета


##Предобработка данных

In [ ]:
# cell 4: Фильтрация съедобных товаров

print("=" * 60)
print("FILTERING EDIBLE PRODUCTS")
print("=" * 60)

# --------------------------------------------------------------------------
# Определение edible department IDs
# --------------------------------------------------------------------------

edible_department_ids = [1, 3, 4, 6, 7, 9, 12, 13, 14, 15, 16, 19, 20]
print(f"Edible department IDs: {edible_department_ids}")

# --------------------------------------------------------------------------
# Определение названий колонок в products
# --------------------------------------------------------------------------

print("\nChecking products table columns:")
print(f"  Columns: {products.columns.tolist()}")

# Определяем колонку для department
if 'department_id' in products.columns:
    dept_col = 'department_id'
elif 'department' in products.columns:
    dept_col = 'department'
else:
    print("  ERROR: No department column found")
    print("  Available columns:", products.columns.tolist())
    raise KeyError("Cannot find department column in products table")

# Определяем колонку для product_id
if 'product_id' in products.columns:
    product_id_col = 'product_id'
elif 'id' in products.columns:
    product_id_col = 'id'
else:
    product_id_col = products.columns[0]

print(f"  Using department column: '{dept_col}'")
print(f"  Using product_id column: '{product_id_col}'")

# --------------------------------------------------------------------------
# Фильтрация каталога товаров
# --------------------------------------------------------------------------

products_edible = products[products[dept_col].isin(edible_department_ids)].copy()

print(f"\nProduct catalog filtering:")
print(f"  Total products: {len(products):,}")
print(f"  Edible products: {len(products_edible):,} ({len(products_edible)/len(products)*100:.1f}%)")

# --------------------------------------------------------------------------
# Определение колонок в order_products
# --------------------------------------------------------------------------

print("\nChecking order_products table columns:")
print(f"  Columns: {order_products.columns.tolist()}")

# Определяем колонку для product_id в order_products
if 'product_id' in order_products.columns:
    op_product_id_col = 'product_id'
elif 'productid' in order_products.columns:
    op_product_id_col = 'productid'
else:
    op_product_id_col = order_products.columns[1] if len(order_products.columns) > 1 else order_products.columns[0]

print(f"  Using product_id column: '{op_product_id_col}'")

# --------------------------------------------------------------------------
# Фильтрация транзакций
# --------------------------------------------------------------------------

edible_product_ids = products_edible[product_id_col].unique()
order_products_edible = order_products[
    order_products[op_product_id_col].isin(edible_product_ids)
].copy()

print(f"\nTransaction filtering:")
print(f"  Prior transactions: {len(order_products):,}")
print(f"  Edible transactions: {len(order_products_edible):,}")

if len(order_products_edible) == 0:
    print("\nWARNING: No edible transactions found!")
    print("Check that product IDs match between products and order_products tables")
    print(f"  Sample product IDs from products: {list(products_edible[product_id_col].head(5))}")
    print(f"  Sample product IDs from order_products: {list(order_products[op_product_id_col].head(5))}")

# --------------------------------------------------------------------------
# Обогащение транзакций данными о товарах (только если есть колонки)
# --------------------------------------------------------------------------

# Определяем доступные колонки для merge
merge_cols = [product_id_col]
if 'product_name' in products_edible.columns:
    merge_cols.append('product_name')
if 'aisle_id' in products_edible.columns or 'aisle' in products_edible.columns:
    aisle_col = 'aisle_id' if 'aisle_id' in products_edible.columns else 'aisle'
    merge_cols.append(aisle_col)
if dept_col in products_edible.columns:
    merge_cols.append(dept_col)

print(f"\nMerging with product info using columns: {merge_cols}")

order_products_edible = order_products_edible.merge(
    products_edible[merge_cols],
    left_on=op_product_id_col,
    right_on=product_id_col,
    how='left'
)

print(f"\nEnriched transactions: {len(order_products_edible):,} rows")
print(f"Final columns: {order_products_edible.columns.tolist()}")

# --------------------------------------------------------------------------
# Статистика по категориям (если есть department колонка)
# --------------------------------------------------------------------------

if dept_col in order_products_edible.columns:
    print("\nEdible products by department:")
    dept_stats = order_products_edible.groupby(dept_col).size().reset_index(name='count')
    dept_stats = dept_stats.sort_values('count', ascending=False)
    for _, row in dept_stats.head(10).iterrows():
        print(f"  Department {row[dept_col]}: {row['count']:,} transactions")
elif 'department_id' in order_products_edible.columns:
    print("\nEdible products by department_id:")
    dept_stats = order_products_edible.groupby('department_id').size().reset_index(name='count')
    dept_stats = dept_stats.sort_values('count', ascending=False)
    for _, row in dept_stats.head(10).iterrows():
        print(f"  Department {row['department_id']}: {row['count']:,} transactions")

print("\n" + "=" * 60)
print("STATUS: Edible products filtering complete")
print("=" * 60)

FILTERING EDIBLE PRODUCTS
Edible department IDs: [1, 3, 4, 6, 7, 9, 12, 13, 14, 15, 16, 19, 20]

Checking products table columns:
  Columns: ['product_id', 'product_name', 'aisle_id', 'department_id']
  Using department column: 'department_id'
  Using product_id column: 'product_id'

Product catalog filtering:
  Total products: 49,688
  Edible products: 35,089 (70.6%)

Checking order_products table columns:
  Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']
  Using product_id column: 'product_id'

Transaction filtering:
  Prior transactions: 32,434,489
  Edible transactions: 30,433,469

Merging with product info using columns: ['product_id', 'product_name', 'aisle_id', 'department_id']

Enriched transactions: 30,433,469 rows
Final columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'product_name', 'aisle_id', 'department_id']

Edible products by department:
  Department 4: 9,479,291 transactions
  Department 16: 5,414,016 transactions
  Department

##Обучающая выборка

In [ ]:
# cell 5: Создание обучающей выборки (оптимизированная версия)

print("=" * 60)
print("CREATING TRAINING DATASET")
print("=" * 60)

N_USERS = 5000
print(f"Number of users to process: {N_USERS}")

# --------------------------------------------------------------------------
# Фильтрация данных
# --------------------------------------------------------------------------

train_orders = orders[orders['eval_set'] == 'prior'].copy()
users = train_orders['user_id'].unique()[:N_USERS]
train_orders = train_orders[train_orders['user_id'].isin(users)]

print(f"Selected orders: {len(train_orders):,}")
print(f"Selected users: {len(users):,}")

# --------------------------------------------------------------------------
# Подготовка данных для быстрого доступа
# --------------------------------------------------------------------------

# Группируем товары по заказам
print("\nPreparing product data...")
order_products_grouped = order_products_edible.groupby('order_id')['product_name'].agg(list).reset_index()
order_products_dict = dict(zip(order_products_grouped['order_id'], order_products_grouped['product_name']))

print(f"  Orders with products: {len(order_products_dict)}")

# --------------------------------------------------------------------------
# Сортировка заказов по пользователям
# --------------------------------------------------------------------------

print("\nProcessing user orders...")
user_order_lists = train_orders.sort_values(['user_id', 'order_number']).groupby('user_id')['order_id'].agg(list).reset_index()
user_order_dict = dict(zip(user_order_lists['user_id'], user_order_lists['order_id']))

# --------------------------------------------------------------------------
# Подготовка данных о заказах для быстрого доступа
# --------------------------------------------------------------------------

orders_info = train_orders.set_index('order_id')[['order_dow', 'order_hour_of_day']].to_dict('index')
has_days_since_prior = 'days_since_prior' in train_orders.columns

if has_days_since_prior:
    days_since_prior_dict = train_orders.set_index('order_id')['days_since_prior'].to_dict()

# --------------------------------------------------------------------------
# Формирование обучающих примеров (оптимизированный цикл)
# --------------------------------------------------------------------------

print("\nCreating training examples...")
training_examples = []
day_map = {0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday',
           4: 'Thursday', 5: 'Friday', 6: 'Saturday'}

for user_id in users:
    order_ids = user_order_dict.get(user_id, [])
    if len(order_ids) < 2:
        continue

    total_orders = len(order_ids)

    for i in range(1, len(order_ids)):
        prev_order_id = order_ids[i - 1]
        current_order_id = order_ids[i]

        # Получаем информацию о заказах
        current_info = orders_info.get(current_order_id, {})
        current_dow = current_info.get('order_dow', 0)
        current_hour = current_info.get('order_hour_of_day', 12)

        # 1. Previous orders
        prev_products = order_products_dict.get(prev_order_id, [])
        previous_orders_text = f"previous orders: {', '.join(prev_products[:10])}"

        # 2. Current context
        current_context_text = f"order placed on {day_map.get(current_dow, 'unknown')} at {current_hour}:00"

        # 3. User data
        if has_days_since_prior:
            avg_days = sum(days_since_prior_dict.get(oid, 0) for oid in order_ids[1:]) / (len(order_ids) - 1)
            user_data_text = f"user history: {total_orders} total orders, average interval {avg_days:.0f} days"
        else:
            user_data_text = f"user history: {total_orders} total orders"

        # 4. Calendar
        calendar_text = f"day: {day_map.get(current_dow, 'unknown')}, hour: {current_hour}"

        # 5. Current cart
        current_products = order_products_dict.get(current_order_id, [])
        cart_text = f"current cart: {', '.join(current_products[:5])}"

        # 6. Target positions (next order)
        next_order_id = order_ids[i + 1] if i + 1 < len(order_ids) else None
        if next_order_id is not None:
            target_products = order_products_dict.get(next_order_id, [])
            target_positions = target_products[:3]
        else:
            target_positions = current_products[:3]

        training_examples.append({
            'user_id': user_id,
            'previous_orders_text': previous_orders_text,
            'current_context_text': current_context_text,
            'user_data_text': user_data_text,
            'calendar_text': calendar_text,
            'cart_text': cart_text,
            'target_positions': target_positions
        })

    # Прогресс
    if len(training_examples) % 10000 == 0 and len(training_examples) > 0:
        print(f"  Processed {len(training_examples):,} examples...")

training_df = pd.DataFrame(training_examples)

print(f"\nTraining examples created: {len(training_df):,}")
print(f"  Unique users: {training_df['user_id'].nunique():,}")

# --------------------------------------------------------------------------
# Статистика
# --------------------------------------------------------------------------

all_targets = []
for targets in training_df['target_positions']:
    all_targets.extend(targets)

if len(all_targets) > 0:
    target_counts = pd.Series(all_targets).value_counts()
    print(f"  Unique target products: {len(target_counts):,}")
    print(f"  Top 5 target products:")
    for product, count in target_counts.head(5).items():
        print(f"    {product[:40]:40} {count:5} occurrences")
else:
    print("  WARNING: No target products found!")

print("\nSample training example:")
print("-" * 40)
if len(training_df) > 0:
    sample_row = training_df.iloc[0]
    print(f"Previous orders: {sample_row['previous_orders_text'][:80]}...")
    print(f"Current context: {sample_row['current_context_text']}")
    print(f"User data: {sample_row['user_data_text']}")
    print(f"Calendar: {sample_row['calendar_text']}")
    print(f"Cart: {sample_row['cart_text'][:80]}...")
    print(f"Target: {sample_row['target_positions']}")
else:
    print("No training examples created. Check data quality.")

print("\n" + "=" * 60)
print("STATUS: Training dataset creation complete")
print("=" * 60)

CREATING TRAINING DATASET
Number of users to process: 5000
Selected orders: 76,832
Selected users: 5,000

Preparing product data...
  Orders with products: 3176918

Processing user orders...

Creating training examples...
  Processed 30,000 examples...

Training examples created: 71,832
  Unique users: 5,000
  Unique target products: 13,809
  Top 5 target products:
    Banana                                    5407 occurrences
    Bag of Organic Bananas                    4603 occurrences
    Organic Strawberries                      1984 occurrences
    Organic Hass Avocado                      1795 occurrences
    Organic Whole Milk                        1685 occurrences

Sample training example:
----------------------------------------
Previous orders: previous orders: Soda, Organic Unsweetened Vanilla Almond Milk, Original Beef Je...
Current context: order placed on Wednesday at 7:00
User data: user history: 10 total orders
Calendar: day: Wednesday, hour: 7
Cart: current cart: Sod

##Обучение

In [ ]:
# cell 6: ГИБРИДНЫЙ RAG (качество эмбеддингов + скорость)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import time
import numpy as np
import pandas as pd
import torch
from collections import Counter

print("=" * 60)
print("HYBRID RAG + RANDOM FOREST (QUALITY + SPEED)")
print("=" * 60)

train_df, test_df = train_test_split(training_df, test_size=0.2, random_state=42)

print(f"\nData split:")
print(f"  Training set: {len(train_df):,} samples")
print(f"  Test set: {len(test_df):,} samples")

# --------------------------------------------------------------------------
# Инициализация (кэширование эмбеддингов на диск)
# --------------------------------------------------------------------------

from sentence_transformers import SentenceTransformer
import faiss
import hashlib
import os
import pickle

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nUsing device: {device}")

# Легкая модель для скорости (в 3 раза быстрее e5)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")

# Директория для кэша в Drive (чтобы не терялся)
cache_dir = '/content/drive/MyDrive/rag_cache/'
os.makedirs(cache_dir, exist_ok=True)

def get_cached_embeddings(texts, cache_name):
    """Кэширование эмбеддингов в Drive (не пропадает после перезапуска)"""
    texts_hash = hashlib.md5(''.join(texts[:100]).encode()).hexdigest()
    cache_path = os.path.join(cache_dir, f'emb_{cache_name}_{texts_hash[:16]}.npy')

    if os.path.exists(cache_path):
        print(f"    Loading from cache: {cache_name}")
        return np.load(cache_path)

    print(f"    Computing embeddings: {cache_name}")
    embeddings = embedding_model.encode(
        texts,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(cache_path, embeddings)
    return embeddings

# --------------------------------------------------------------------------
# Построение FAISS индексов (только для train)
# --------------------------------------------------------------------------

print("\nBuilding FAISS indices (train only)...")

sources = ['previous_orders', 'current_context', 'user_data', 'calendar', 'cart']
indices = {}
source_embeddings = {}

for source in sources:
    print(f"  {source}...")
    col_name = f'{source}_text'
    texts = train_df[col_name].fillna('').tolist()

    if len(texts) == 0:
        indices[source] = None
        continue

    embeddings = get_cached_embeddings(texts, f"{source}_train")
    source_embeddings[source] = embeddings

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype('float32'))
    indices[source] = index

# --------------------------------------------------------------------------
# Извлечение RAG признаков (с FAISS поиском)
# --------------------------------------------------------------------------

print("\nExtracting RAG features (with FAISS search)...")

def extract_rag_features(df, indices, k=5):
    """Настоящие RAG признаки: эмбеддинг запроса + сходство + контекст"""
    all_features = []

    for source in sources:
        if indices.get(source) is None:
            continue

        col_name = f'{source}_text'
        queries = df[col_name].fillna('').tolist()

        # Эмбеддинги запросов одним батчем
        query_embs = embedding_model.encode(
            queries,
            batch_size=128,
            show_progress_bar=False,
            normalize_embeddings=True
        ).astype('float32')

        source_features = []
        index = indices[source]

        # FAISS поиск батчами
        batch_size = 500
        for i in range(0, len(query_embs), batch_size):
            batch = query_embs[i:i+batch_size]
            similarities, retrieved_indices = index.search(batch, k)

            for j, (emb, sims, ret_idx) in enumerate(zip(batch, similarities, retrieved_indices)):
                valid_sims = sims[sims > -1]
                max_sim = valid_sims.max() if len(valid_sims) > 0 else 0
                mean_sim = valid_sims.mean() if len(valid_sims) > 0 else 0

                # Признак 1: эмбеддинг запроса (384)
                feat = emb.tolist()
                # Признак 2: метрики сходства (2)
                feat.extend([max_sim, mean_sim])
                # Признак 3: средний эмбеддинг найденных документов (384)
                # Берем только top-1 документ для скорости
                if ret_idx[0] != -1 and source_embeddings.get(source) is not None:
                    retrieved_emb = source_embeddings[source][ret_idx[0]]
                    feat.extend(retrieved_emb.tolist())
                else:
                    feat.extend([0] * embedding_dim)

                source_features.append(feat)

        all_features.append(np.array(source_features))

    # Объединяем все источники
    return np.hstack(all_features)

X_train = extract_rag_features(train_df, indices)
X_test = extract_rag_features(test_df, indices)

print(f"\n  Training features shape: {X_train.shape}")
print(f"  Test features shape: {X_test.shape}")
# (384 эмбеддинг + 2 метрики + 384 retrieved) * 5 источников = ~3850 признаков

# --------------------------------------------------------------------------
# Балансировка классов (улучшает редкие классы)
# --------------------------------------------------------------------------

print("\nPreparing balanced target labels...")
label_encoder = LabelEncoder()

train_targets = ['|'.join(t) if isinstance(t, list) else str(t) for t in train_df['target_positions']]
test_targets = ['|'.join(t) if isinstance(t, list) else str(t) for t in test_df['target_positions']]

# Оставляем классы с >= 5 примерами для стабильности
counter = Counter(train_targets)
min_samples = 5
top_classes = [c for c, count in counter.items() if count >= min_samples]

print(f"  Classes with >= {min_samples} samples: {len(top_classes)} (from {len(counter)})")

train_mask = [t in top_classes for t in train_targets]
test_mask = [t in top_classes for t in test_targets]

X_train = X_train[train_mask]
X_test = X_test[test_mask]

y_train = np.array([train_targets[i] for i in range(len(train_targets)) if train_mask[i]])
y_test = np.array([test_targets[i] for i in range(len(test_targets)) if test_mask[i]])

y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print(f"  Training samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")

# --------------------------------------------------------------------------
# Стандартизация
# --------------------------------------------------------------------------

print("\nStandardizing...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------------------------------------
# Random Forest с балансировкой
# --------------------------------------------------------------------------

print("\nTraining Random Forest (balanced)...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight='balanced',  # ключевое улучшение для редких классов
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
rf_model.fit(X_train_scaled, y_train_enc)
train_time = time.time() - start_time

print(f"\n  Training completed in {train_time:.2f} seconds")

# --------------------------------------------------------------------------
# Оценка
# --------------------------------------------------------------------------

print("\nEvaluating on test set...")
y_pred = rf_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test_enc, y_pred)

print(f"\nTest accuracy: {accuracy:.4f}")

# Подробный отчет по основным классам
print("\nClassification report (weighted):")
print(classification_report(y_test_enc, y_pred, zero_division=0))

# --------------------------------------------------------------------------
# Сохранение
# --------------------------------------------------------------------------

import joblib

save_path = '/content/drive/MyDrive/rag_order_model_hybrid'
os.makedirs(save_path, exist_ok=True)

joblib.dump(rf_model, f"{save_path}/rf_model.pkl")
joblib.dump(label_encoder, f"{save_path}/label_encoder.pkl")
joblib.dump(scaler, f"{save_path}/scaler.pkl")
joblib.dump(indices, f"{save_path}/indices.pkl")
joblib.dump(sources, f"{save_path}/sources.pkl")

print(f"\nModel saved to {save_path}")

print("\n" + "=" * 60)
print("STATUS: Training complete")
print("=" * 60)

HYBRID RAG + RANDOM FOREST (QUALITY + SPEED)

Data split:
  Training set: 57,465 samples
  Test set: 14,367 samples

Using device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  calendar...
    Computing embeddings: calendar_train


Batches:   0%|          | 0/449 [00:00<?, ?it/s]

  cart...
    Computing embeddings: cart_train


Batches:   0%|          | 0/449 [00:00<?, ?it/s]


Extracting RAG features (with FAISS search)...


In [ ]:
# cell 7: Инференс и предсказания

import joblib
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

print("=" * 60)
print("INFERENCE: ORDER PREDICTION")
print("=" * 60)

# --------------------------------------------------------------------------
# Загрузка модели
# --------------------------------------------------------------------------

save_path = '/content/drive/MyDrive/rag_order_model_edible'

print("\nLoading trained model...")

# Загружаем компоненты
rf_model = joblib.load(f"{save_path}/rf_model.pkl")
label_encoder = joblib.load(f"{save_path}/label_encoder.pkl")
scaler = joblib.load(f"{save_path}/scaler.pkl")

# Загружаем данные для инференса
with open(f"{save_path}/inference_data.pkl", 'rb') as f:
    inference_data = joblib.load(f)

# Загружаем FAISS индексы
indices = {}
for source in ['previous_orders', 'current_context', 'user_data', 'calendar', 'cart']:
    index_path = f"{save_path}/faiss_{source}.bin"
    if os.path.exists(index_path):
        indices[source] = faiss.read_index(index_path)
    else:
        indices[source] = None

# Загружаем embedding model
embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')
embedding_dim = embedding_model.get_sentence_embedding_dimension()

print(f"Model loaded successfully")
print(f"  Classes available: {len(label_encoder.classes_)}")

# --------------------------------------------------------------------------
# Функции для предсказания
# --------------------------------------------------------------------------

def predict(customer, top_k=5):
    """Предсказание для одного клиента"""
    features = []

    for source in ['previous_orders', 'current_context', 'user_data', 'calendar', 'cart']:
        query_text = customer.get(source, '')

        if not query_text or indices.get(source) is None:
            features.extend([0] * (embedding_dim + 2))
            continue

        # Эмбеддинг запроса
        query_emb = embedding_model.encode([query_text], normalize_embeddings=True).astype('float32')

        # Поиск в FAISS
        similarities, _ = indices[source].search(query_emb, k=5)

        valid_sims = similarities[0][similarities[0] > -1]
        max_sim = valid_sims.max() if len(valid_sims) > 0 else 0
        mean_sim = valid_sims.mean() if len(valid_sims) > 0 else 0

        # Объединяем эмбеддинг и метрики
        feat = np.concatenate([query_emb[0], [max_sim, mean_sim]])
        features.extend(feat)

    # Стандартизация
    X = np.array(features).reshape(1, -1)
    X_scaled = scaler.transform(X)

    # Предсказание
    pred_encoded = rf_model.predict(X_scaled)[0]
    pred_label = label_encoder.inverse_transform([pred_encoded])[0]

    if '|' in pred_label:
        return pred_label.split('|')[:top_k]
    return [pred_label]

def predict_with_proba(customer, top_k=5):
    """Предсказание с вероятностями"""
    features = []

    for source in ['previous_orders', 'current_context', 'user_data', 'calendar', 'cart']:
        query_text = customer.get(source, '')

        if not query_text or indices.get(source) is None:
            features.extend([0] * (embedding_dim + 2))
            continue

        query_emb = embedding_model.encode([query_text], normalize_embeddings=True).astype('float32')
        similarities, _ = indices[source].search(query_emb, k=5)

        valid_sims = similarities[0][similarities[0] > -1]
        max_sim = valid_sims.max() if len(valid_sims) > 0 else 0
        mean_sim = valid_sims.mean() if len(valid_sims) > 0 else 0

        feat = np.concatenate([query_emb[0], [max_sim, mean_sim]])
        features.extend(feat)

    X = np.array(features).reshape(1, -1)
    X_scaled = scaler.transform(X)

    probas = rf_model.predict_proba(X_scaled)[0]
    top_indices = np.argsort(probas)[-top_k:][::-1]

    results = []
    for idx in top_indices:
        label = label_encoder.inverse_transform([idx])[0]
        if '|' in label:
            for product in label.split('|')[:top_k]:
                results.append((product, probas[idx] / len(label.split('|'))))
        else:
            results.append((label, probas[idx]))

    # Агрегируем вероятности
    aggregated = {}
    for product, prob in results:
        aggregated[product] = max(aggregated.get(product, 0), prob)

    return sorted(aggregated.items(), key=lambda x: x[1], reverse=True)[:top_k]

# --------------------------------------------------------------------------
# Примеры
# --------------------------------------------------------------------------

customer_1 = {
    'previous_orders': 'milk, bread, eggs, butter, cheese',
    'current_context': 'evening dinner preparation for family of four',
    'user_data': 'family with two children, weekly grocery shopper',
    'calendar': 'Friday evening, weekend ahead',
    'cart': 'pizza dough, tomato sauce, mozzarella'
}

print("\n" + "-" * 40)
print("Example 1: New customer prediction")
print("-" * 40)

predictions = predict(customer_1, top_k=5)
print(f"\nPredicted products (top 5):")
for i, product in enumerate(predictions):
    print(f"  {i+1}. {product}")

customer_2 = {
    'previous_orders': 'coffee, creamer, sugar, cookies',
    'current_context': 'morning coffee break',
    'user_data': 'single professional, daily coffee drinker',
    'calendar': 'Monday morning, start of work week',
    'cart': 'coffee beans, milk'
}

print("\n" + "-" * 40)
print("Example 2: Prediction with probabilities")
print("-" * 40)

predictions_proba = predict_with_proba(customer_2, top_k=5)
print(f"\nPredicted products with probabilities:")
for i, (product, prob) in enumerate(predictions_proba):
    print(f"  {i+1}. {product:40} {prob:.2%}")

customers = [
    {
        'previous_orders': 'pizza, soda, chips',
        'current_context': 'fast dinner',
        'user_data': 'student',
        'calendar': 'Friday night',
        'cart': 'frozen pizza'
    },
    {
        'previous_orders': 'salad, vegetables, fruits',
        'current_context': 'healthy lunch',
        'user_data': 'fitness enthusiast',
        'calendar': 'Wednesday afternoon',
        'cart': 'quinoa, avocado'
    }
]

print("\n" + "-" * 40)
print("Example 3: Batch inference")
print("-" * 40)

for i, customer in enumerate(customers):
    pred = predict(customer, top_k=3)
    print(f"\nCustomer {i+1}:")
    print(f"  Context: {customer['current_context']}")
    print(f"  Predictions: {', '.join(pred)}")

print("\n" + "=" * 60)
print("STATUS: Inference complete")
print("=" * 60)